# 03 — Polymarket Signals

Demonstrates the Polymarket prediction-market signal pipeline:

1. **Market search** — Gamma API with tag-slug inference and client-side filtering
2. **YES-bias correction** — Reichenbach & Walther (2025) additive discount
3. **Quality weighting** — age, liquidity, tier, whale penalty, transitory discount
4. **Conditional return** — expected asset return from adjusted probabilities
5. **Signal aggregation** — quality-weighted average across markets
6. **History flags** — price spikes (whale proxy) and transitory moves

**Prerequisites:**
```bash
conda activate cramer-research
cd ../research && pip install -e . && cd ../notebooks
```

In [ ]:
import os
import sys

sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from research.data.polymarket import fetch_polymarket_markets
from research.models.ensemble import (
    MarketInput,
    adjust_yes_bias,
    compute_market_quality,
    compute_conditional_return,
    compute_polymarket_signal,
)
from research.utils.calibration import adjust_yes_bias as utils_adjust_yes_bias

## 1. Search Polymarket Markets

Fetch prediction markets for a given query. The Gamma API `keyword` param is non-functional — we use tag-slug inference and client-side text filtering instead.

In [ ]:
QUERY = "bitcoin"
LIMIT = 15

markets_df = fetch_polymarket_markets(QUERY, limit=LIMIT, min_volume_24h=1000)
print(f"Found {len(markets_df)} markets for '{QUERY}'")
markets_df[['question', 'probability', 'volume_24h', 'age_days']].head(10)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
if not markets_df.empty:
    ax.scatter(markets_df['volume_24h'], markets_df['probability'] * 100, s=80, alpha=0.6)
    ax.set_xlabel('24h Volume (USD)')
    ax.set_ylabel('YES Probability (%)')
    ax.set_title(f"Polymarket Markets — {QUERY} (volume vs probability)")
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
else:
    print("No markets found")

## 2. YES-Bias Correction

Reichenbach & Walther (2025) found systematic YES-overtrading across 124M Polymarket trades.
Apply additive discount: `p_adj = p - β` when `p > 0.5` (default β = 0.035 = 3.5pp).

In [ ]:
# Demonstrate bias correction on a range of raw probabilities
raw_probs = np.linspace(0.05, 0.95, 20)
adjusted_probs = [adjust_yes_bias(p) for p in raw_probs]

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(raw_probs * 100, raw_probs * 100, 'k--', label='No correction')
ax.plot(raw_probs * 100, np.array(adjusted_probs) * 100, linewidth=2, label='YES-bias corrected')
ax.axvline(50, color='gray', linestyle=':', alpha=0.5)
ax.set_xlabel('Raw YES Probability (%)')
ax.set_ylabel('Adjusted Probability (%)')
ax.set_title('YES-Bias Correction (Reichenbach & Walther 2025)')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 3. Market Quality Weights

Each market gets a quality score based on:
- `w_age = min(1, age_days / 21)` — mature markets > 21 days get full weight
- `w_liq = min(1, log10(volume + 1) / 6)` — higher volume = more reliable
- `tau` — tier discount: macro=0.90, geopolitical=0.75, electoral=0.55
- Whale penalty: `-50%` if price spike detected
- Transitory discount: `-30%` if 24-48h move reversed

In [ ]:
# Convert DataFrame to MarketInput objects
market_inputs = []
for _, row in markets_df.iterrows():
    m = MarketInput(
        question=row['question'],
        probability=row['probability'],
        volume24h_usd=row['volume_24h'],
        age_days=int(row['age_days']) if pd.notna(row['age_days']) else None,
        signal_tier='macro',  # assume macro for crypto
        delta_yes=0.06,       # example: +6% if YES
        delta_no=-0.04,       # example: -4% if NO
    )
    market_inputs.append(m)

# Compute quality for each market
qualities = [compute_market_quality(m) for m in market_inputs]
for i, (m, q) in enumerate(zip(market_inputs[:5], qualities[:5])):
    print(f"{m.question[:60]:60s}  p={m.probability:.3f}  w={q:.3f}")

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
if qualities:
    ax.barh(range(len(qualities)), qualities, color='steelblue')
    ax.set_yticks(range(len(market_inputs)))
    ax.set_yticklabels([m.question[:50] for m in market_inputs], fontsize=8)
    ax.set_xlabel('Quality Weight')
    ax.set_title(f'Market Quality Weights — {QUERY}')
    ax.grid(True, alpha=0.3, axis='x')
    plt.tight_layout()
    plt.show()
else:
    print("No quality scores computed")

## 4. Conditional Return

For each market, compute expected return: `E[r] = p_adj × deltaYes + (1 - p_adj) × deltaNo`

In [ ]:
returns = []
for m in market_inputs:
    p_adj = adjust_yes_bias(m.probability)
    r = compute_conditional_return(p_adj, m.delta_yes, m.delta_no)
    returns.append(r)

result_df = pd.DataFrame({
    'question': [m.question[:60] for m in market_inputs],
    'probability': [m.probability for m in market_inputs],
    'p_adjusted': [adjust_yes_bias(m.probability) for m in market_inputs],
    'quality': qualities,
    'conditional_return': returns,
})

print(result_df.head(10).to_string(index=False))

## 5. Aggregate Polymarket Signal

Quality-weighted average of conditional returns across all markets.

In [ ]:
pm_result = compute_polymarket_signal(market_inputs)

print(f"PM Signal:         {pm_result['signal']:.4f}  ({pm_result['signal']*100:.2f}%)")
print(f"Avg Quality:       {pm_result['avg_quality']:.4f}")
print(f"Warnings:          {len(pm_result['warnings'])}")
for w in pm_result['warnings'][:5]:
    print(f"  - {w}")

## 6. History Flags — Whale & Transitory Detection

Simulate history flags to show how they impact quality weights.

- **Price spike** (whale proxy): `|P_now - P_2h_ago| > 0.08` AND volume < $100k
- **Transitory move**: 24-48h move > 10pp AND reversed > 50% toward baseline

In [ ]:
# Create a synthetic market with whale flag
whale_market = MarketInput(
    question="Will Bitcoin exceed $90k by end of month?",
    probability=0.72,
    volume24h_usd=50_000,  # low volume
    age_days=14,
    price_spike_detected=True,  # whale proxy
    signal_tier='macro',
    delta_yes=0.08,
    delta_no=-0.05,
)

# Same market without whale flag
normal_market = MarketInput(
    question="Will Bitcoin exceed $90k by end of month?",
    probability=0.72,
    volume24h_usd=50_000,
    age_days=14,
    price_spike_detected=False,
    signal_tier='macro',
    delta_yes=0.08,
    delta_no=-0.05,
)

q_whales = compute_market_quality(whale_market)
q_normal = compute_market_quality(normal_market)

print(f"Normal market quality:     {q_normal:.4f}")
print(f"Whale-flag market quality: {q_whales:.4f}")
print(f"Penalty:                   {q_normal - q_whales:.4f}  ({100*(1 - q_whales/q_normal):.1f}% reduction)")

In [ ]:
# Compare transitory flag
transitory_market = MarketInput(
    question="Will Bitcoin exceed $90k by end of month?",
    probability=0.72,
    volume24h_usd=50_000,
    age_days=14,
    transitory_move=True,
    signal_tier='macro',
    delta_yes=0.08,
    delta_no=-0.05,
)

q_transitory = compute_market_quality(transitory_market)
print(f"Transitory-flag quality:   {q_transitory:.4f}")
print(f"Penalty:                   {q_normal - q_transitory:.4f}  ({100*(1 - q_transitory/q_normal):.1f}% reduction)")

## 7. Signal Decomposition

Break down the Polymarket signal by market contribution.

In [ ]:
contributions = []
for m in market_inputs:
    p_adj = adjust_yes_bias(m.probability)
    w = compute_market_quality(m)
    r = compute_conditional_return(p_adj, m.delta_yes, m.delta_no)
    contributions.append({
        'question': m.question[:50],
        'weight': w,
        'return': r,
        'contribution': w * r,
    })

contrib_df = pd.DataFrame(contributions)

if contrib_df.empty:
    print('No contributions to plot')
else:
    contrib_df = contrib_df.sort_values('contribution', ascending=False)
    fig, ax = plt.subplots(figsize=(10, 5))
    colors = ['green' if c > 0 else 'red' for c in contrib_df['contribution']]
    ax.barh(range(len(contrib_df)), contrib_df['contribution'] * 100, color=colors, alpha=0.7)
    ax.set_yticks(range(len(contrib_df)))
    ax.set_yticklabels(contrib_df['question'], fontsize=8)
    ax.set_xlabel('Contribution (%)')
    ax.set_title(f"Polymarket Signal Decomposition — {QUERY}")
    ax.grid(True, alpha=0.3, axis='x')
    plt.tight_layout()
    plt.show()


## 8. Sensitivity: Volume Threshold

How does the signal change with different minimum volume filters?

In [ ]:
volume_thresholds = [0, 1000, 5000, 10_000, 50_000, 100_000]
signals = []

for thresh in volume_thresholds:
    df = fetch_polymarket_markets(QUERY, limit=LIMIT, min_volume_24h=thresh)
    if len(df) == 0:
        signals.append(0)
        continue
    inputs = []
    for _, row in df.iterrows():
        inputs.append(MarketInput(
            question=row['question'],
            probability=row['probability'],
            volume24h_usd=row['volume_24h'],
            age_days=int(row['age_days']) if pd.notna(row['age_days']) else None,
            signal_tier='macro',
            delta_yes=0.06,
            delta_no=-0.04,
        ))
    result = compute_polymarket_signal(inputs)
    signals.append(result['signal'])

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(volume_thresholds, np.array(signals) * 100, marker='o', linewidth=2)
ax.axhline(0, color='gray', linestyle='--', alpha=0.5)
ax.set_xlabel('Min Volume Threshold (USD)')
ax.set_ylabel('PM Signal (%)')
ax.set_title(f'{QUERY}: Signal Sensitivity to Volume Filter')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()